In [1]:
import pandas as pd
import numpy as np

In [3]:
merged = pd.read_csv('covid_merged_clean.csv', parse_dates=['Date'])


Building on the cleaned and merged dataset from `01_data_cleaning.ipynb`. Computing
four standard epidemiological ratios (growth rate, test positivity rate, case
fatality ratio, recovery rate) per state, per day, to support later trend analysis
and risk classification.

In [7]:
# Sort by State and Date first
merged = merged.sort_values(['State', 'Date']).reset_index(drop=True)


In [8]:
# 1. Growth rate: (Cases_today - Cases_yesterday) / Cases_yesterday, per state
merged['Cases_prev_day'] = merged.groupby('State')['Confirmed'].shift(1)
merged['Growth_Rate'] = (merged['Confirmed'] - merged['Cases_prev_day']) / merged['Cases_prev_day']
print(merged['Growth_Rate'].replace([np.inf, -np.inf], np.nan).describe())

count    17966.000000
mean         0.031009
std          0.228971
min         -1.000000
25%          0.000858
50%          0.006436
75%          0.022926
max         14.000000
Name: Growth_Rate, dtype: float64


Growth rate:


*   mean = 3.1% -> on average, cases grew 3.1% day-over-day across the whole timeline
*   min = -1 -> a day where cases dropped by 100% relative to the day before
*   max = 14 -> a day where cases were 14x the previous day
* median = 0.6% -> a typical day saw 0.6% growth



In [9]:
# 2. Test positivity rate: Positive / TotalSamples (both cumulative in this dataset)
merged['Positivity_Rate'] = merged['Positive'] / merged['TotalSamples']
print(merged['Positivity_Rate'].describe())

count    5590.000000
mean        0.044281
std         0.040741
min         0.000000
25%         0.017177
50%         0.031747
75%         0.058730
max         0.219857
Name: Positivity_Rate, dtype: float64


Positivity Rate:


*   mean = 4.4% -> on average, 4.4% of tests came back positive
*   max = 21.9 -> worst single day had ~22% positivity
* median = 3.17% -> typical day saw ~3% of tests as positive



In [10]:
# 3. Case fatality ratio
merged['CFR'] = merged['Deaths'] / merged['Confirmed']
print(merged['CFR'].describe())

count    18002.000000
mean         0.013180
std          0.016307
min          0.000000
25%          0.005355
50%          0.011943
75%          0.016339
max          0.500000
Name: CFR, dtype: float64


Case Fatality Rate (CFR):


*   mean = 1.3% -> 1.3% of confirmed cases resulted in death
*   max = 50% -> worst day statistic showed 50% of confirmed cases resulting in death



In [11]:
# 4. Recovery rate
merged['Recovery_Rate'] = merged['Cured'] / merged['Confirmed']
print(merged['Recovery_Rate'].describe())

count    18002.000000
mean         0.773241
std          0.269930
min          0.000000
25%          0.685818
50%          0.892070
75%          0.971354
max          1.000000
Name: Recovery_Rate, dtype: float64


Recovery Rate:


*   mean = 77.3% -> on average ~77% confirmed cases had recovered by the time of each row
*   median = 89.2%
* max = 1 = 100% recovery



In [12]:
# Save results to csv file
merged.to_csv('covid_with_ratios.csv', index=False)